# Northwind – Exploratory Analysis & Hypothesis Testing

This notebook connects to the `dw` star schema (MySQL), explores the sales data and tests several business hypotheses.
Run the pipeline first (`python run_pipeline.py`) and set `DB_USER` / `DB_PASSWORD` in your environment or `.env` file.

**Contents:** 1. Data access · 2. Correlation · 3. ANOVA by category · 4. Discount hypotheses · 5. Category post-hoc tests · 6. Country vs category (chi-square) · 7. Conclusions

## 1. Data access

In [14]:
import sys
import pandas as pd
sys.path.insert(0, "../src")
from db_connector import get_db_engine


engine = get_db_engine()   # credentials come from environment variables (see src/config.py)

df_sales = pd.read_sql("SELECT * FROM dw.fact_orders LIMIT 5;", con=engine)

display(df_sales)

,order_id,customer_sk,product_sk,employee_sk,supplier_sk,date_key,unit_price,quantity,discount,line_total
0,10248,85,11,5,5,19960704,14.0,12,0.0,168.0
1,10248,85,42,5,20,19960704,9.8,10,0.0,98.0
2,10248,85,72,5,14,19960704,34.8,5,0.0,174.0
3,10249,79,14,6,6,19960705,18.6,9,0.0,167.4
4,10249,79,51,6,24,19960705,42.4,40,0.0,1696.0


In [ ]:
import sys
import pandas as pd
sys.path.insert(0, "../src")
from db_connector import get_db_engine

engine = get_db_engine()   # credentials come from environment variables (see src/config.py)


query = """
SELECT 
    f.order_id,
    f.quantity,
    f.unit_price,
    f.discount,
    f.line_total,
    d.product_name,
    d.category_name,
    c.country AS customer_country
FROM dw.fact_orders f
JOIN dw.dim_products d ON f.product_sk = d.product_sk
JOIN dw.dim_customers c ON f.customer_sk = c.customer_sk
LIMIT 10;
"""

df_analysis = pd.read_sql(query, con=engine)
display(df_analysis)

,order_id,quantity,unit_price,discount,line_total,product_name,category_name,customer_country
0,10285,45,14.4,0.20,518.4,Chai,Beverages,Germany
1,10294,18,14.4,0.00,259.2,Chai,Beverages,USA
2,10317,20,14.4,0.00,288.0,Chai,Beverages,USA
3,10348,15,14.4,0.15,183.6,Chai,Beverages,Germany
4,10354,12,14.4,0.00,172.8,Chai,Beverages,Mexico
5,10370,15,14.4,0.15,183.6,Chai,Beverages,Switzerland
6,10406,10,14.4,0.00,144.0,Chai,Beverages,Brazil
7,10413,24,14.4,0.00,345.6,Chai,Beverages,France
8,10477,15,14.4,0.00,216.0,Chai,Beverages,Portugal
9,10522,40,18.0,0.20,576.0,Chai,Beverages,Germany


In [18]:
import sys
import pandas as pd
sys.path.insert(0, "../src")
from db_connector import get_db_engine

# Միացում dw բազային
engine = get_db_engine()   # credentials come from environment variables (see src/config.py)

# Հարցում՝ վերլուծության համար անհրաժեշտ բոլոր չափերով ու ֆակտերով
query = """
SELECT 
    f.order_id,
    f.quantity,
    f.unit_price,
    f.discount,
    f.line_total,
    d.category_name,
    c.country AS customer_country
FROM dw.fact_orders f
JOIN dw.dim_products d ON f.product_sk = d.product_sk
JOIN dw.dim_customers c ON f.customer_sk = c.customer_sk;
"""

df = pd.read_sql(query, con=engine)

# Ստուգենք առաջին մի քանի տողերը և տվյալների տիպերը
print("Տվյալների չափը (Rows, Columns):", df.shape)
display(df.head())

Տվյալների չափը (Rows, Columns): (2155, 7)


,order_id,quantity,unit_price,discount,line_total,category_name,customer_country
0,10285,45,14.4,0.20,518.4,Beverages,Germany
1,10294,18,14.4,0.00,259.2,Beverages,USA
2,10317,20,14.4,0.00,288.0,Beverages,USA
3,10348,15,14.4,0.15,183.6,Beverages,Germany
4,10354,12,14.4,0.00,172.8,Beverages,Mexico


## 2. Correlation analysis
How do quantity, price and discount relate to the line total?

In [19]:
correlation_matrix = df[
    ['quantity', 'unit_price', 'discount', 'line_total']
].corr()


display(correlation_matrix)

,quantity,unit_price,discount,line_total
quantity,1.000000,0.007366,0.128577,0.479889
unit_price,0.007366,1.000000,-0.011166,0.703493
discount,0.128577,-0.011166,1.000000,0.006565
line_total,0.479889,0.703493,0.006565,1.000000


## 3. One-way ANOVA – does product category affect line value?
* **H0:** mean `line_total` is the same in every product category.
* **H1:** at least one category has a different mean `line_total`.

In [22]:
import scipy.stats as stats

# 1. Group data by product category for line_total
categories = df['category_name'].unique()
group_data = [
    df[df['category_name'] == cat]['line_total'].dropna() for cat in categories
]

# 2. Perform One-way ANOVA test
f_stat, p_value = stats.f_oneway(*group_data)

print(f'ANOVA F-statistic: {f_stat:.4f}')
print(f'P-value: {p_value:.4e}')

# 3. Summarize results
if p_value < 0.05:
  print(
      'Result is statistically significant (Reject Null Hypothesis):'
      ' Categories have a significant effect on line_total.'
  )
else:
  print(
      'No statistically significant difference (Fail to reject Null'
      ' Hypothesis).'
  )

ANOVA F-statistic: 7.1982
P-value: 1.5368e-08
Result is statistically significant (Reject Null Hypothesis): Categories have a significant effect on line_total.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import scipy.stats as stats
sns.set_theme(style="whitegrid")

## 4. Discounts – do they change what customers buy?

**Hypothesis A (quantity)**
* H0: mean quantity per order line is equal for discounted and non-discounted lines.
* H1: the means differ.

**Hypothesis B (revenue)**
* H0: mean `line_total` is equal for discounted and non-discounted lines.
* H1: the means differ.

Welch's t-test (unequal variances) is paired with the non-parametric Mann-Whitney U test because the data are right-skewed.

In [ ]:
disc = df[df['discount'] > 0]
none = df[df['discount'] == 0]

for metric in ['quantity', 'line_total']:
    a, b = disc[metric], none[metric]
    t, p_t = stats.ttest_ind(a, b, equal_var=False)
    u, p_u = stats.mannwhitneyu(a, b, alternative='two-sided')
    diff = a.mean() - b.mean()
    se = np.sqrt(a.var() / len(a) + b.var() / len(b))
    print(f"--- {metric} ---")
    print(f"mean discounted = {a.mean():.2f} | mean no discount = {b.mean():.2f} | difference = {diff:.2f}")
    print(f"95% CI of difference: [{diff - 1.96*se:.2f}, {diff + 1.96*se:.2f}]")
    print(f"Welch t = {t:.3f}, p = {p_t:.3g} | Mann-Whitney p = {p_u:.3g}\n")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
plot_df = df.assign(discounted=np.where(df['discount'] > 0, 'Discounted', 'No discount'))
sns.boxplot(data=plot_df, x='discounted', y='quantity', ax=axes[0], showfliers=False)
axes[0].set_title('Quantity per order line')
sns.boxplot(data=plot_df, x='discounted', y='line_total', ax=axes[1], showfliers=False)
axes[1].set_title('Line total per order line')
plt.tight_layout(); plt.show()

**Interpretation (expected from the data):**
* *Quantity:* discounted lines average about 27.1 units vs 21.7 without a discount. The 95% CI of the difference is roughly [3.7, 7.1] units and both tests give p < 0.001, so we **reject H0** – discounts are associated with larger quantities.
* *Revenue:* the average line total is about $615 (discounted) vs $570 (not), but the difference is **not significant** (Welch p ≈ 0.30; Mann-Whitney p ≈ 0.07), so we **fail to reject H0**. Discounts raise volume but not revenue per line, i.e. the extra units are bought at a lower price.

## 5. Which categories differ? (Kruskal-Wallis + Tukey HSD)
Line totals are right-skewed, so the ANOVA result is checked with the non-parametric Kruskal-Wallis test, and Tukey HSD shows which pairs of categories differ.

In [ ]:
groups = {name: g['line_total'].values for name, g in df.groupby('category_name')}
h, p_kw = stats.kruskal(*groups.values())
print(f"Kruskal-Wallis H = {h:.2f}, p = {p_kw:.3g}")

tukey = stats.tukey_hsd(*groups.values())
names = list(groups)
sig = [(names[i], names[j], tukey.pvalue[i, j])
       for i in range(len(names)) for j in range(i + 1, len(names)) if tukey.pvalue[i, j] < 0.05]
print("\nSignificantly different category pairs (Tukey HSD, p < 0.05):")
for a, b, p in sorted(sig, key=lambda x: x[2]):
    print(f"  {a} vs {b}: p = {p:.4f}")

summary = df.groupby('category_name')['line_total'].agg(['mean', 'median']).sort_values('mean', ascending=False)
summary.plot(kind='bar', figsize=(10, 4), title='Line total by category: mean vs median')
plt.ylabel('USD'); plt.tight_layout(); plt.show()

**Interpretation (expected from the data):** Kruskal-Wallis agrees with the ANOVA (p < 0.001). Meat/Poultry has the highest mean line value (about $942) and differs significantly from most other categories; Seafood has the lowest (about $398). For Beverages the mean ($663) is far above the median ($252), meaning a few very large lines (premium products) pull the average up – use the median when describing a "typical" Beverages order line.

## 6. Chi-square test – is category mix independent of customer country?
* **H0:** product category and customer country (top 5 countries by revenue) are independent.
* **H1:** category mix depends on the country.

In [ ]:
top_countries = df.groupby('customer_country')['line_total'].sum().nlargest(5).index
sub = df[df['customer_country'].isin(top_countries)]
table = pd.crosstab(sub['customer_country'], sub['category_name'])

chi2, p, dof, expected = stats.chi2_contingency(table)
print(f"chi2 = {chi2:.2f}, dof = {dof}, p = {p:.3f}")
print(f"cells with expected count < 5: {(expected < 5).sum()}")

(table.div(table.sum(axis=1), axis=0) * 100).round(1).plot(
    kind='bar', stacked=True, figsize=(10, 4), title='Category mix of order lines by country (%)')
plt.ylabel('% of order lines'); plt.legend(bbox_to_anchor=(1.01, 1), loc='upper left'); plt.tight_layout(); plt.show()

**Interpretation (expected from the data):** chi2 ≈ 30.7, dof = 28, p ≈ 0.33 – we **fail to reject H0**. In the top-5 markets customers buy a similar category mix, so there is no evidence for country-specific assortments; regional differences are in *volume*, not in *what* is bought.

## 7. Conclusions & recommendations
1. **Price drives line value** (r = 0.70 with `unit_price`), quantity second (r = 0.48); discount is almost uncorrelated with line total (r = 0.01).
2. **Category matters** for line value (ANOVA and Kruskal-Wallis, p < 0.001); Meat/Poultry lines are the most valuable, Seafood the least.
3. **Discounts lift quantity (about +5 units per line) but not revenue per line** (p ≈ 0.30). Before extending discounts, check whether the extra volume covers the lower margin.
4. **Category mix is similar across the top markets** (p ≈ 0.33): one assortment strategy can serve them, and growth should come from customer volume.